# TALOS — Reindex de Embeddings (Colab Offload)

**Corpus:** Corpus completo | **Modelo:** Qwen/Qwen3-Embedding-8B | **Output:** vectors_*.jsonl o vector_store.db

## Configuración:
- Modelo de alta fidelidad: `Qwen/Qwen3-Embedding-8B` (4096 dimensiones)
- Batch size: 160 con GPU
- L2 Normalization activada (`normalize_embeddings=True`)


In [ ]:
from google.colab import drive
drive.mount("/content/drive")
print("Drive montado")

In [ ]:
import os, glob

DRIVE_FOLDER = "/content/drive/MyDrive/TALOS"  # Carpeta principal de TALOS en Drive
OUTPUT_DB    = f"{DRIVE_FOLDER}/vector_store.db"
SHARD_DIR    = "/content/vec_shards"

os.makedirs(SHARD_DIR, exist_ok=True)
print(f"Directorio de trabajo: {DRIVE_FOLDER}")


In [ ]:
!pip install -q sentence-transformers torch tqdm


In [ ]:
import torch
print("CUDA disponible:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


In [ ]:
import sqlite3
con = sqlite3.connect(f"file:{KNOWLEDGE_DB}?mode=ro", uri=True)
rows = con.execute("SELECT id, source_id, content FROM chunks ORDER BY id").fetchall()
chunks = [(r[0], r[1], r[2] or "") for r in rows]
src_to_nbs = {}
for src_id, nb_id in con.execute("SELECT source_id, notebook_id FROM notebook_sources").fetchall():
    src_to_nbs.setdefault(src_id, []).append(nb_id)
con.close()
print(f"Chunks: {len(chunks):,} | Fuentes: {len(src_to_nbs):,}")

In [ ]:
from sentence_transformers import SentenceTransformer

MODEL_NAME = "Qwen/Qwen3-Embedding-8B"
DIMENSIONS = 4096
BATCH_SIZE = 160

print(f"Cargando {MODEL_NAME} en {DEVICE}...")
model = SentenceTransformer(MODEL_NAME, device=DEVICE, trust_remote_code=True)

def embed_passages(texts):
    embeddings = model.encode(
        texts,
        batch_size=BATCH_SIZE,
        normalize_embeddings=True,
        show_progress_bar=False,
    )
    return embeddings.tolist() if hasattr(embeddings, "tolist") else [list(e) for e in embeddings]

test_vec = embed_passages(["test de conexion semantica TALOS"])[0]
assert len(test_vec) == DIMENSIONS, f"Error dimension: {len(test_vec)} != {DIMENSIONS}"
print(f"Smoke test OK: {DIMENSIONS} dimensiones")


In [ ]:
import struct, time, json
from tqdm.notebook import tqdm

PROVIDER_ID = "sentence_transformers"
MODEL_ID    = "Qwen3-Embedding-8B"

# Buscar el archivo pending más reciente en DRIVE_FOLDER
pending_files = sorted(glob.glob(f"{DRIVE_FOLDER}/pending_*.jsonl"))
if pending_files:
    latest_pending = pending_files[-1]
    print(f"Procesando pending: {latest_pending}")
    records_to_embed = []
    with open(latest_pending, "r", encoding="utf-8") as fh:
        for line in fh:
            if not line.strip(): continue
            obj = json.loads(line)
            if "_meta" in obj: continue
            records_to_embed.append(obj)
    print(f"Total chunks a vectorizar: {len(records_to_embed):,}")
    
    out_vec_file = f"{DRIVE_FOLDER}/vectors_{time.strftime('%Y%m%dT%H%M%SZ')}.jsonl"
    with open(out_vec_file, "w", encoding="utf-8") as out_fh:
        for i in tqdm(range(0, len(records_to_embed), BATCH_SIZE), desc="Embedding Qwen3"):
            batch = records_to_embed[i:i+BATCH_SIZE]
            texts = [r.get("text", "") for r in batch]
            vecs = embed_passages(texts)
            for r, vec in zip(batch, vecs):
                out_fh.write(json.dumps({
                    "chunk_id": r["chunk_id"],
                    "notebook_id": r["notebook_id"],
                    "vector": vec
                }) + "\n")
    print(f"Vectores guardados en Drive: {out_vec_file}")


In [ ]:
final_conn = sqlite3.connect(OUTPUT_DB)
final_conn.execute("PRAGMA journal_mode = WAL;")
final_conn.execute("PRAGMA synchronous = NORMAL;")
final_conn.execute("DROP TABLE IF EXISTS chunk_embeddings;")
final_conn.execute("""CREATE TABLE chunk_embeddings (
    chunk_id TEXT NOT NULL, notebook_id TEXT NOT NULL, source_id TEXT NOT NULL,
    provider_id TEXT NOT NULL, model_id TEXT NOT NULL, dimensions INTEGER NOT NULL,
    vector BLOB NOT NULL, PRIMARY KEY (chunk_id, notebook_id));""")
final_conn.execute("CREATE INDEX IF NOT EXISTS idx_vec_notebook ON chunk_embeddings(notebook_id);")
final_conn.execute("CREATE INDEX IF NOT EXISTS idx_vec_source ON chunk_embeddings(source_id);")
final_conn.execute(f"ATTACH DATABASE '{shard_path}' AS shard0")
final_conn.execute("BEGIN")
final_conn.execute("INSERT OR REPLACE INTO chunk_embeddings SELECT chunk_id, notebook_id, source_id, provider_id, model_id, dimensions, vector FROM shard0.chunk_embeddings")
final_conn.commit()
final_conn.execute("DETACH DATABASE shard0")
counts = final_conn.execute("SELECT notebook_id, COUNT(*) FROM chunk_embeddings GROUP BY notebook_id ORDER BY 2 DESC").fetchall()
total_final = final_conn.execute("SELECT COUNT(*) FROM chunk_embeddings").fetchone()[0]
final_conn.close()
size_mb = os.path.getsize(OUTPUT_DB) / 1e6
print(f"vector_store.db: {size_mb:.1f} MB | {total_final:,} filas")
for nb, n in counts: print(f"  {nb}: {n:,}")

## Para reemplazar localmente:



O si usas Drive Sync, simplemente copiá el archivo desde la carpeta sincronizada.